In [ ]:
import time
import requests

# Safe standalone alert helper (run after the main pipeline cell defines `extreme_fires`)
if "extreme_fires" not in globals() or "DISCORD_WEBHOOK_URL" not in globals():
    print("⚠️ Run the main pipeline cell first to populate `extreme_fires` and `DISCORD_WEBHOOK_URL`.")
else:
    for _, row in extreme_fires.head(2).iterrows():
        payload = {
            "content": "🔴 **[RT - REAL-TIME STREAM] NASA FIRMS WILDFIRE ALERT** 🔴",
            "embeds": [{
                "title": "Live Thermal Anomaly Telemetry",
                "color": 15158332,
                "fields": [
                    {"name": "Status", "value": "🟢 RT Active Stream", "inline": True},
                    {"name": "Risk Tier", "value": str(row["risk_tier"]), "inline": True},
                    {"name": "Brightness (RT)", "value": str(row["brightness"]), "inline": True},
                    {"name": "Coordinates", "value": f"Lat: {row['latitude']}, Lon: {row['longitude']}", "inline": False},
                    {"name": "Processed At", "value": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()), "inline": False}
                ]
            }]
        }
        requests.post(DISCORD_WEBHOOK_URL, json=payload, timeout=10)
        time.sleep(1)

In [ ]:
import json
import sqlite3
import time
from datetime import datetime, timezone
from typing import Optional

import folium
import openpyxl
import pandas as pd
import requests
from pydantic import BaseModel, Field

# --- STEP 1: FETCH LIVE TELEMETRY FROM NASA FIRMS ---
MAP_KEY = "11be2b95414f1e231600092cc3ff7148"
source = "VIIRS_NOAA20_NRT"
area_coords = "world"
day_range = 1
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{source}/{area_coords}/{day_range}"

print("Step 1: Fetching live telemetry from NASA FIRMS...")
live_df = pd.read_csv(api_url)
print(f"-> Fetched {len(live_df)} live wildfire/thermal records.")

# --- STEP 2: PYDANTIC VALIDATION MODEL & RISK SCORING ---
class WildfireRecord(BaseModel):
    latitude: float = Field(..., ge=-90, le=90)
    longitude: float = Field(..., ge=-180, le=180)
    brightness: float
    acq_date: str
    risk_tier: Optional[str] = "Moderate"

def assign_risk_tier(bright: float) -> str:
    if bright > 350:
        return "Extreme"
    if bright > 325:
        return "High"
    if bright > 300:
        return "Moderate"
    return "Low"

print("Step 2: Running Pydantic validation and risk scoring...")
bright_col = "bright_ti4" if "bright_ti4" in live_df.columns else "brightness"
validated_rows = []

for _, row in live_df.head(200).iterrows():
    try:
        lat = float(row["latitude"])
        lon = float(row["longitude"])
        bright = float(row[bright_col])
        date_str = str(row["acq_date"])

        record = WildfireRecord(
            latitude=lat,
            longitude=lon,
            brightness=bright,
            acq_date=date_str,
        )
        data_dict = record.model_dump()
        data_dict["risk_tier"] = assign_risk_tier(bright)
        validated_rows.append(data_dict)
    except Exception:
        continue

validated_df = pd.DataFrame(validated_rows)
print(f"-> Successfully validated {len(validated_df)} records.")

if not validated_df.empty:
    # --- STEP 3: SAVE INTO SQLITE DATABASE ---
    print("Step 3: Storing validated records into SQLite database...")
    conn = sqlite3.connect("wildfire_pipeline.db")
    try:
        validated_df.to_sql("wildfire_events", conn, if_exists="replace", index=False)
    finally:
        conn.close()
    print("-> Saved successfully to 'wildfire_pipeline.db'.")

    # --- STEP 4: GENERATE ESRI SATELLITE FOLIUM MAP ---
    print("Step 4: Generating live geospatial Esri satellite map...")
    map_center = [float(validated_df["latitude"].mean()), float(validated_df["longitude"].mean())]
    live_map = folium.Map(location=map_center, zoom_start=3, tiles="Esri.WorldImagery")

    color_map = {"Extreme": "red", "High": "orange", "Moderate": "yellow", "Low": "green"}
    for _, row in validated_df.iterrows():
        tier = row["risk_tier"]
        color = color_map.get(tier, "blue")
        folium.CircleMarker(
            location=[float(row["latitude"]), float(row["longitude"])],
            radius=6,
            popup=f"Risk: {tier}<br>Brightness: {row['brightness']} K<br>Date: {row['acq_date']}",
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
        ).add_to(live_map)

    live_map.fit_bounds([
        [float(validated_df["latitude"].min()), float(validated_df["longitude"].min())],
        [float(validated_df["latitude"].max()), float(validated_df["longitude"].max())],
    ])
    print("-> Live Map generated successfully!")

    # --- STEP 5: AUTOMATED MULTI-FORMAT ANALYTICS & BATCH REPORTING ---
    print("Step 5: Generating analytical JSON, CSV, and Excel export files...")
    total_events = int(len(validated_df))
    avg_brightness = round(float(validated_df["brightness"].mean()), 2)
    tier_counts = validated_df["risk_tier"].value_counts().to_dict()

    now_utc = datetime.now(timezone.utc)
    timestamp_str = now_utc.strftime("%Y%m%d_%H%M%S")

    json_filename = f"wildfire_report_{timestamp_str}.json"
    report_data = {
        "execution_timestamp_utc": now_utc.strftime("%Y-%m-%d %H:%M:%S"),
        "total_records_processed": total_events,
        "average_brightness_kelvin": avg_brightness,
        "risk_tier_breakdown": tier_counts,
    }
    with open(json_filename, "w", encoding="utf-8") as f:
        json.dump(report_data, f, indent=4)
    print(f"-> JSON report saved: '{json_filename}'")

    csv_filename = f"wildfire_dataset_{timestamp_str}.csv"
    validated_df.to_csv(csv_filename, index=False)
    print(f"-> CSV dataset saved: '{csv_filename}'")

    excel_filename = f"wildfire_dataset_{timestamp_str}.xlsx"
    validated_df.to_excel(excel_filename, index=False, sheet_name="Wildfire Telemetry")
    print(f"-> Excel spreadsheet saved: '{excel_filename}'")

    # --- STEP 6: DISCORD WEBHOOK ALERTS FOR EXTREME EVENTS ---
    DISCORD_WEBHOOK_URL = "https://discord.com/api/webhooks/1559237316734619689/rRGo1yahxxLI0pccKnbr-uPhqxx6Pojhef8Ll2ZQTd2bsk-aTSpiIQvKQ4LEKKY82058"
    extreme_fires = validated_df[validated_df["risk_tier"] == "Extreme"]
    print(f"Step 6: Scanning for Extreme risks... Found {len(extreme_fires)} extreme event(s).")

    for _, row in extreme_fires.head(10).iterrows():
        payload = {
            "content": "🔴 **[RT - REAL-TIME STREAM] NASA FIRMS WILDFIRE ALERT** 🔴",
            "embeds": [{
                "title": "Live Thermal Anomaly Telemetry",
                "color": 15158332,
                "fields": [
                    {"name": "Status", "value": "🟢 RT Active Stream", "inline": True},
                    {"name": "Risk Tier", "value": str(row["risk_tier"]), "inline": True},
                    {"name": "Batch Avg Brightness", "value": f"{avg_brightness} K", "inline": True},
                    {"name": "Brightness (Event)", "value": str(row["brightness"]), "inline": True},
                    {"name": "Coordinates", "value": f"Lat: {row['latitude']}, Lon: {row['longitude']}", "inline": False},
                    {"name": "Processed At", "value": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()), "inline": False}
                ]
            }]
        }
        response = requests.post(DISCORD_WEBHOOK_URL, json=payload, timeout=10)
        if response.status_code in (200, 204):
            print(f" -> Alert sent to Discord! (Lat: {row['latitude']}, Lon: {row['longitude']})")
        else:
            print(f" -> Failed to send alert. Status: {response.status_code} | Reason: {response.text}")
        time.sleep(1)

    print("\n✅ FULL LIVE PIPELINE & ANALYTICS EXECUTION COMPLETE!")
else:
    print("Warning: No records passed validation. Check raw dataframe columns.")

In [ ]:
%%writefile app.py
import json
import os
import sqlite3

import folium
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import streamlit as st
from streamlit_folium import st_folium

sns.set_theme(style="whitegrid")

st.set_page_config(
    page_title="NASA FIRMS Wildfire Monitoring Dashboard",
    page_icon="🔥",
    layout="wide",
)

st.title("🔥 NASA FIRMS Real-Time Global Wildfire Monitoring Dashboard")
st.markdown("Production-grade data engineering pipeline tracking thermal anomalies, risk tiers, and telemetry.")

@st.cache_data
def load_data():
    if not os.path.exists("wildfire_pipeline.db"):
        return pd.DataFrame()
    conn = sqlite3.connect("wildfire_pipeline.db")
    try:
        df = pd.read_sql("SELECT * FROM wildfire_events", conn)
    finally:
        conn.close()
    return df

df = load_data()

if df.empty:
    st.warning("⚠️ No wildfire data found in 'wildfire_pipeline.db'. Please run your master pipeline cell first to populate the database!")
else:
    st.sidebar.header("Dashboard Controls")
    selected_risk = st.sidebar.selectbox(
        "Filter by Risk Tier",
        ["All", "Extreme", "High", "Moderate", "Low"]
    )

    filtered_df = df[df["risk_tier"] == selected_risk] if selected_risk != "All" else df

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Records Processed", len(df))
    col2.metric("Average Brightness", f"{round(df['brightness'].mean(), 2)} K")
    col3.metric("Extreme Risk Count", int((df["risk_tier"] == "Extreme").sum()))
    col4.metric("High Risk Count", int((df["risk_tier"] == "High").sum()))

    st.markdown("----")
    map_col, table_col = st.columns([1.2, 1])

    with map_col:
        st.subheader("🌍 Geospatial Satellite Map (Esri Imagery)")
        if filtered_df.empty:
            st.info("No records match the selected risk tier.")
        else:
            map_center = [float(filtered_df["latitude"].mean()), float(filtered_df["longitude"].mean())]
            live_map = folium.Map(location=map_center, zoom_start=3, tiles="Esri.WorldImagery")

            for _, row in filtered_df.iterrows():
                tier = row["risk_tier"]
                color = "red" if tier == "Extreme" else "orange" if tier == "High" else "yellow" if tier == "Moderate" else "green"
                folium.CircleMarker(
                    location=[float(row["latitude"]), float(row["longitude"])],
                    radius=6,
                    popup=f"Risk: {tier}<br>Brightness: {row['brightness']}<br>Date: {row['acq_date']}",
                    color=color,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.8,
                ).add_to(live_map)

            st_folium(live_map, use_container_width=True, height=450)

    with table_col:
        st.subheader("📊 Filtered Telemetry Records")
        st.dataframe(filtered_df[["latitude", "longitude", "brightness", "risk_tier", "acq_date"]], use_container_width=True, height=450)

    st.markdown("----")
    st.subheader("📈 Statistical EDA & Visual Reports")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sns.histplot(df["brightness"], kde=True, ax=axes[0], color="crimson")
    axes[0].set_title("Brightness Temperature Distribution")
    axes[0].set_xlabel("Brightness (Kelvin)")
    axes[0].set_ylabel("Frequency")

    risk_order = ["Extreme", "High", "Moderate", "Low"]
    sns.countplot(
        x="risk_tier",
        hue="risk_tier",
        data=df,
        ax=axes[1],
        order=[r for r in risk_order if r in df["risk_tier"].unique()],
        palette="rocket",
        legend=False,
    )
    axes[1].set_title("Thermal Anomalies by Risk Tier")
    axes[1].set_xlabel("Risk Tier")
    axes[1].set_ylabel("Record Count")

    if {"latitude", "brightness"}.issubset(df.columns):
        sns.scatterplot(x="latitude", y="brightness", hue="risk_tier", data=df, ax=axes[2], palette="deep")
        axes[2].set_title("Brightness vs. Latitude Correlation")
        axes[2].set_xlabel("Latitude (°)")
        axes[2].set_ylabel("Brightness (Kelvin)")
    else:
        axes[2].text(0.5, 0.5, "Data Unavailable", horizontalalignment="center", verticalalignment="center")
        axes[2].set_title("Correlation Plot Unavailable")

    plt.tight_layout()
    st.pyplot(fig)

    st.markdown("----")
    st.subheader("📥 Export & Reports")
    col_dl1, col_dl2 = st.columns(2)

    with col_dl1:
        st.download_button(
            label="Download Full Dataset as CSV",
            data=df.to_csv(index=False).encode("utf-8"),
            file_name="wildfire_export.csv",
            mime="text/csv",
        )

    with col_dl2:
        report_summary = {
            "total_records": int(len(df)),
            "average_brightness": round(float(df["brightness"].mean()), 2),
            "risk_breakdown": df["risk_tier"].value_counts().to_dict(),
        }
        st.download_button(
            label="Download Summary Report as JSON",
            data=json.dumps(report_summary, indent=4),
            file_name="wildfire_report.json",
            mime="application/json",
        )

In [ ]:
import sqlite3
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

conn = sqlite3.connect("wildfire_pipeline.db")
try:
    df = pd.read_sql("SELECT * FROM wildfire_events", conn)
finally:
    conn.close()

if df.empty:
    print("⚠️ No data found in 'wildfire_pipeline.db'. Please run your ingestion pipeline cell first!")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sns.histplot(df["brightness"], kde=True, ax=axes[0], color="crimson")
    axes[0].set_title("Brightness Temperature Distribution")
    axes[0].set_xlabel("Brightness (Kelvin)")
    axes[0].set_ylabel("Frequency")

    risk_order = ["Extreme", "High", "Moderate", "Low"]
    sns.countplot(
        x="risk_tier",
        hue="risk_tier",
        data=df,
        ax=axes[1],
        order=[r for r in risk_order if r in df["risk_tier"].unique()],
        palette="rocket",
        legend=False,
    )
    axes[1].set_title("Thermal Anomalies by Risk Tier")
    axes[1].set_xlabel("Risk Tier")
    axes[1].set_ylabel("Record Count")

    if {"latitude", "brightness"}.issubset(df.columns):
        sns.scatterplot(x="latitude", y="brightness", hue="risk_tier", data=df, ax=axes[2], palette="deep")
        axes[2].set_title("Brightness vs. Latitude Correlation")
        axes[2].set_xlabel("Latitude (°)")
        axes[2].set_ylabel("Brightness (Kelvin)")
    else:
        axes[2].text(0.5, 0.5, "Data Unavailable", horizontalalignment="center", verticalalignment="center")
        axes[2].set_title("Correlation Plot Unavailable")

    plt.tight_layout()
    plt.show()

    print("\n📊 --- Statistical Summary for EDA Report ---")
    avail_cols = [col for col in ["brightness", "latitude", "longitude"] if col in df.columns]
    print(df[avail_cols].describe())

In [ ]:
import os
import subprocess
import sys
import time

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "streamlit", "streamlit-folium", "openpyxl"],
    check=True,
)

print("🚀 Starting Streamlit server...")
streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.headless", "true"]
)
time.sleep(4)

tunnel_cmd = "./cloudflared" if os.path.exists("./cloudflared") else "cloudflared"
print("🌐 Opening Cloudflare public tunnel...")
subprocess.run([tunnel_cmd, "tunnel", "--url", "http://localhost:8501"], check=False)